# Question 1: Word Embeddings (Word2Vec) *(30 points)*

Word2Vec is a popular technique in natural language processing (NLP) for learning vector representations of words, also known as word embeddings. These embeddings capture semantic relationships between words, such that words used in similar contexts are placed close to each other in the vector space.

There are two main architectures for Word2Vec:

- **Continuous Bag-of-Words (CBOW):** Predicts the current word based on its context (surrounding words).
- **Skip-Gram:** Predicts surrounding words given the current word.

In this tutorial, you will implement a simplified version of the Skip-Gram model using Python and train it on a given dataset.

---

## Your Task:

#### **Data Preparation:**

- Given a corpus of text (word2vec_dataset.en) attached, preprocess the data to create training examples suitable for a Skip-Gram model.
- Implement functions to tokenize the text, build a vocabulary, and generate input-output pairs for training.

#### **Implement a Simple Skip-Gram Model:**

- implement a simple neural network representing the Skip-Gram architecture.
- The model should learn word embeddings by training on the generated input-output pairs.

#### **Train the Model:**

- Train your Skip-Gram model on the prepared data.
- Use appropriate loss functions and optimization algorithms.

#### **Visualize Word Embeddings:**

- After training, visualize the learned word embeddings in 2D space using techniques like PCA or t-SNE.
- Plot the words to observe how similar words are positioned relative to each other.


#### **Notes:**

- If the training process is slow, consider reducing the size of the dataset to improve training speed.

### Allowed Libraries:
- **collections** for building vocabulary.
- **NumPy:** For numerical computations.
- **Matplotlib:** For plotting and visualization.
- **scikit-learn (sklearn):** For dimensionality reduction techniques like PCA or t-SNE, evaluation metrics.
- **TensorFlow or PyTorch** if you prefer to implement the model using these frameworks. However, since the challenge aims to be simple and educational, using NumPy suffices.




In [ ]:
import string
from collections import Counter
import nltk
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# Text bereinigen & tokenisieren
def clean_and_tokenize(text):
    text = text.lower()
    # Satzzeichen entfernen
    for p in string.punctuation:
        text = text.replace(p, " ")
    # Ziffern entfernen
    text = ''.join([ch for ch in text if not ch.isdigit()])
    # In Wörter splitten
    tokens = text.split()
    return tokens

# Datei einlesen
def load_corpus_tokens(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()
    tokens = clean_and_tokenize(text)
    print(f"Anzahl Tokens im Korpus: {len(tokens)}")
    print("Erste 20 Tokens:", tokens[:20])
    return tokens

# Vokabular erstellen
def build_vocabulary(tokens, min_freq=2, max_vocab_size=5000):
    counter = Counter(tokens)
    # Filter nach Häufigkeit
    if min_freq > 1:
        counter = Counter({w: c for w, c in counter.items() if c >= min_freq})
    # Nach Häufigkeit sortieren und begrenzen
    most_common = counter.most_common(max_vocab_size)
    word_to_id = {w: i for i, (w, _) in enumerate(most_common)}
    id_to_word = [w for w, _ in most_common]
    print(f"Vokabulargröße: {len(word_to_id)}")
    return word_to_id, id_to_word

# Tokens in IDs umwandeln
def tokens_to_ids(tokens, word_to_id):
    ids = []
    for w in tokens:
        if w in word_to_id:
            ids.append(word_to_id[w])
    return ids

def generate_skipgram_pairs(corpus_ids, window_size=2):         
    pairs = []  # alle (center, context) Paare sammeln
    # mit Index über alle Positionen in der ID-Liste durchgehen
    for center_pos in range(len(corpus_ids)):
        center_id = corpus_ids[center_pos]  # ID des zentralen Wortes
        start = max(0, center_pos - window_size)
        end = min(len(corpus_ids) - 1, center_pos + window_size)
        #  durch alle Positionen im Kontextfenster durchgehen
        for context_pos in range(start, end + 1):
            # Das zentrale Wort selbst überspringen
            if context_pos == center_pos:
                continue
            context_id = corpus_ids[context_pos]
            # Ein Trainingspaar (center, context) hinzufügen
            pairs.append((center_id, context_id))
    return pairs

def softmax(x):  # Wandelt einen Vektor von Scores in Wahrscheinlichkeiten um die sich zu 1 aufsummieren
    x_shifted = x - np.max(x)
    exp_x = np.exp(x_shifted)
    return exp_x / np.sum(exp_x)

def initialize_embeddings(vocab_size, embedding_dim):
    np.random.seed(42)  
    # Kleine zufällige Werte
    W_in = 0.01 * np.random.randn(vocab_size, embedding_dim)
    W_out = 0.01 * np.random.randn(embedding_dim, vocab_size)
    return W_in, W_out

def train_skipgram(         
    training_pairs,
    vocab_size,
    embedding_dim=50,
    learning_rate=0.05,
    epochs=3 ):
    W_in, W_out = initialize_embeddings(vocab_size, embedding_dim)
    #  reduzieren, damit es schneller geht
    max_pairs = 50000 
    pairs = training_pairs[:max_pairs]
    print(f"Wir trainieren vorerst auf {len(pairs)} Paaren.")
    for epoch in range(epochs):         
        total_loss = 0.0
        np.random.shuffle(pairs)
        for center_id, context_id in pairs:
            # Embedding des Center-Worts 
            h = W_in[center_id]
            # Scores für alle Wörter 
            scores = np.dot(h, W_out)
            # Softmax 
            y_pred = softmax(scores)
            # + kleine Konstante, um log(0) zu vermeiden
            loss = -np.log(y_pred[context_id] + 1e-10)
            total_loss += loss
            grad_scores = y_pred.copy()
            grad_scores[context_id] -= 1.0  
            grad_W_out = np.outer(h, grad_scores)
            grad_h = np.dot(W_out, grad_scores)
            #  Parameter-Update (Gradientenabstieg)
            W_out -= learning_rate * grad_W_out
            W_in[center_id] -= learning_rate * grad_h
        avg_loss = total_loss / len(pairs)
        print(f"Epoch {epoch+1}/{epochs} - durchschnittlicher Loss: {avg_loss:.4f}")
    return W_in, W_out

def visualize_embeddings(W_in, id_to_word, num_words=100):
    vocab_size, embedding_dim = W_in.shape
    # Falls num_words größer als Vokabular ist, begrenzen
    num_words = min(num_words, vocab_size)
    # Embeddings der ersten num_words Wörter auswählen
    embeddings_subset = W_in[:num_words]
    # PCA auf 2 Dimensionen
    pca = PCA(n_components=2)
    embeddings_2d = pca.fit_transform(embeddings_subset)
    x_coords = embeddings_2d[:, 0]
    y_coords = embeddings_2d[:, 1]
    plt.figure(figsize=(10, 8))
    plt.scatter(x_coords, y_coords)
    # Jedes Wort als Textlabel einzeichnen
    for i in range(num_words):
        word = id_to_word[i]
        plt.annotate(word, (x_coords[i], y_coords[i]))
    plt.title("Word Embeddings (PCA, 2D)")
    plt.xlabel("Dimension 1")
    plt.ylabel("Dimension 2")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    filepath = "/content/word2vec_dataset.en"
    # Datei einlesen
    corpus_tokens = load_corpus_tokens(filepath)
    # Vokabular bauen (angepasst!)
    word_to_id, id_to_word = build_vocabulary(corpus_tokens, min_freq=2, max_vocab_size=5000)
    # Tokens in IDs umwandeln
    corpus_ids = tokens_to_ids(corpus_tokens, word_to_id)
    print("\nBeispiel-Tokens:", corpus_tokens[:20])
    print("Beispiel-IDs   :", corpus_ids[:20])
    # Skip-Gram-Trainingspaare erzeugen
    window_size = 2
    training_pairs = generate_skipgram_pairs(corpus_ids, window_size=window_size)
    print(f"\nAnzahl Skip-Gram-Trainingspaare gesamt: {len(training_pairs)}")
    print("Beispielpaare (erste 5):")
    for pair in training_pairs[:5]:
        center_id, context_id = pair
        print(f"({id_to_word[center_id]} -> {id_to_word[context_id]})")
    # Skip-Gram-Modell trainieren
    embedding_dim = 50  # z.B. 50-dimensionale Embeddings
    W_in, W_out = train_skipgram(
        training_pairs,
        vocab_size=len(word_to_id),
        embedding_dim=embedding_dim,
        learning_rate=0.05,
        epochs=3
    )
    # Embedding für ein bestimmtes Wort anzeigen
    example_word = "car"
    if example_word in word_to_id:
        idx = word_to_id[example_word]
        print(f"\nEmbedding für Wort '{example_word}':")
        print(W_in[idx])
    else:
        print(f"\nDas Wort '{example_word}' ist nicht im Vokabular.")
    # Embeddings visualisieren
    visualize_embeddings(W_in, id_to_word, num_words=100)


# Question 2: Transformer model *(70 points)*

As a Machine Learning engineer at a tech company, you were given a task to develop a machine translation system that translates **English (source) to German (Target)**. You have the freedom to select any dataset for training the model. Use a small subset of data as a validation dataset and report the BLEU score on the validation set.

Also, provide a short description of your transformer model architecture, hyperparameters, and training (also provide the train-validation loss curve). Write your findings and analysis in paragraphs.


**Dataset**

Here are some of the parallel datasets (see Datasets and Resources file):
* Europarl Parallel corpus - https://www.statmt.org/europarl/v7/de-en.tgz
* News Commentary - https://www.statmt.org/wmt14/training-parallel-nc-v9.tgz (use DE-EN parallel data)
* Common Crawl corpus - https://www.statmt.org/wmt13/training-parallel-commoncrawl.tgz (use DE-EN parallel data)

You can also use other datasets of your choice. In this case please add the dataset you used to the submission zip file.

In the above datasets, **'.en'** file has the text in English, and **'.de'** file contains their corresponding German translations.



## Notes:
1. You can also consider using a small subset of the dataset if the training dataset is large
2. Sometimes you can also get out of memory errors while training, so choose the hyperparameters carefully.
3. Your training will be much faster if you use a GPU (Edit -> Notebook settings). If you are using a CPU, it may take several hours or even days. (you can also use Google Colab GPUs for training. link: https://colab.research.google.com/)
4. It is a best practise to leverage vector representations learned in Q1 in your training as starting point for the embedding layer.




In [ ]:
import math
import random
from pathlib import Path
from collections import Counter
import warnings

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

import nltk
from nltk.translate.bleu_score import corpus_bleu
import matplotlib.pyplot as plt


warnings.filterwarnings("ignore", category=UserWarning)

nltk.download("punkt")  # für BLEU 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


EN_FILE = Path("/content/europarl-v7.de-en.en")
DE_FILE = Path("/content/europarl-v7.de-en.de")

assert EN_FILE.exists(), f"Englische Datei nicht gefunden: {EN_FILE}"
assert DE_FILE.exists(), f"Deutsche Datei nicht gefunden: {DE_FILE}"

#  zeilenweise einlesen
with EN_FILE.open(encoding="utf-8") as f_en, DE_FILE.open(encoding="utf-8") as f_de:
    en_sentences = [line.strip() for line in f_en]
    de_sentences = [line.strip() for line in f_de]

print("Anzahl Zeilen EN:", len(en_sentences))
print("Anzahl Zeilen DE:", len(de_sentences))

# Sicherstellen, dass beides gleich lang ist
num_pairs = min(len(en_sentences), len(de_sentences))
en_sentences = en_sentences[:num_pairs]
de_sentences = de_sentences[:num_pairs]

print("Verwendete Satzpaare:", num_pairs)
print("Beispiel:")
print("EN:", en_sentences[0])
print("DE:", de_sentences[0])

# Nur einen Teil des Datensatzes (für schnelleres Training)
MAX_PAIRS = 25000  
num_pairs_small = min(MAX_PAIRS, len(en_sentences))
en_sentences = en_sentences[:num_pairs_small]
de_sentences = de_sentences[:num_pairs_small]

print("Genutzte Anzahl Satzpaare:", num_pairs_small)

# Train/Val-Split
VAL_RATIO = 0.1
num_val = int(num_pairs_small * VAL_RATIO)
num_train = num_pairs_small - num_val

indices = list(range(num_pairs_small))
random.seed(42)
random.shuffle(indices)

train_indices = indices[:num_train]
val_indices = indices[num_train:]

train_en = [en_sentences[i] for i in train_indices]
train_de = [de_sentences[i] for i in train_indices]

val_en = [en_sentences[i] for i in val_indices]
val_de = [de_sentences[i] for i in val_indices]

print(f"Train-Sätze: {len(train_en)}")
print(f"Val-Sätze:   {len(val_en)}")

# Tokenisierung und Vokabular

def simple_tokenize(text):
    return text.lower().strip().split()

PAD_TOKEN = "<pad>"
SOS_TOKEN = "<sos>"
EOS_TOKEN = "<eos>"
UNK_TOKEN = "<unk>"
SPECIAL_TOKENS = [PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN]

def build_vocab(sentences, max_size=20000, min_freq=2):
    counter = Counter()
    for s in sentences:
        tokens = simple_tokenize(s)
        counter.update(tokens)

    most_common = [w for w, c in counter.items() if c >= min_freq]
    most_common = sorted(most_common, key=lambda w: counter[w], reverse=True)
    if max_size is not None:
        most_common = most_common[: max_size - len(SPECIAL_TOKENS)]

    itos = SPECIAL_TOKENS + most_common
    stoi = {w: i for i, w in enumerate(itos)}
    return stoi, itos

src_stoi, src_itos = build_vocab(train_en, max_size=20000, min_freq=2)  # Englisch
tgt_stoi, tgt_itos = build_vocab(train_de, max_size=20000, min_freq=2)  # Deutsch

print("Vokabulargröße EN:", len(src_itos))
print("Vokabulargröße DE:", len(tgt_itos))

PAD_IDX = src_stoi[PAD_TOKEN]
SOS_IDX = src_stoi[SOS_TOKEN]
EOS_IDX = src_stoi[EOS_TOKEN]
UNK_IDX = src_stoi[UNK_TOKEN]

def encode_sentence(text, stoi, add_sos_eos=False):
    tokens = simple_tokenize(text)
    ids = [stoi.get(tok, stoi[UNK_TOKEN]) for tok in tokens]
    if add_sos_eos:
        return [stoi[SOS_TOKEN]] + ids + [stoi[EOS_TOKEN]]
    else:
        return ids

# Dataset & DataLoader

class TranslationDataset(Dataset):
    def __init__(self, src_sentences, tgt_sentences, src_stoi, tgt_stoi, max_len=100):
        assert len(src_sentences) == len(tgt_sentences)
        self.src_sentences = src_sentences
        self.tgt_sentences = tgt_sentences
        self.src_stoi = src_stoi
        self.tgt_stoi = tgt_stoi
        self.max_len = max_len

    def __len__(self):
        return len(self.src_sentences)

    def __getitem__(self, idx):
        src_text = self.src_sentences[idx]
        tgt_text = self.tgt_sentences[idx]

        src_ids = encode_sentence(src_text, self.src_stoi, add_sos_eos=False)
        tgt_ids = encode_sentence(tgt_text, self.tgt_stoi, add_sos_eos=True)

        src_ids = src_ids[: self.max_len]
        tgt_ids = tgt_ids[: self.max_len]

        tgt_input  = tgt_ids[:-1]
        tgt_output = tgt_ids[1:]

        return (
            torch.tensor(src_ids, dtype=torch.long),
            torch.tensor(tgt_input, dtype=torch.long),
            torch.tensor(tgt_output, dtype=torch.long),
        )

def collate_fn(batch):
    src_seqs, tgt_in_seqs, tgt_out_seqs = zip(*batch)

    src_lens = [len(s) for s in src_seqs]
    tgt_lens = [len(s) for s in tgt_in_seqs]

    max_src_len = max(src_lens)
    max_tgt_len = max(tgt_lens)

    padded_src = []
    padded_tgt_in = []
    padded_tgt_out = []

    for src, tgt_in, tgt_out in zip(src_seqs, tgt_in_seqs, tgt_out_seqs):
        pad_len_src = max_src_len - len(src)
        padded_src.append(
            torch.cat([src, torch.full((pad_len_src,), src_stoi[PAD_TOKEN], dtype=torch.long)])
        )

        pad_len_tgt_in = max_tgt_len - len(tgt_in)
        padded_tgt_in.append(
            torch.cat([tgt_in, torch.full((pad_len_tgt_in,), tgt_stoi[PAD_TOKEN], dtype=torch.long)])
        )

        pad_len_tgt_out = max_tgt_len - len(tgt_out)
        padded_tgt_out.append(
            torch.cat([tgt_out, torch.full((pad_len_tgt_out,), tgt_stoi[PAD_TOKEN], dtype=torch.long)])
        )

    src_batch = torch.stack(padded_src)        # (B, S)
    tgt_in_batch = torch.stack(padded_tgt_in)  # (B, T)
    tgt_out_batch = torch.stack(padded_tgt_out) # (B, T)

    return src_batch, tgt_in_batch, tgt_out_batch

BATCH_SIZE = 32

train_dataset = TranslationDataset(train_en, train_de, src_stoi, tgt_stoi, max_len=50)
val_dataset   = TranslationDataset(val_en, val_de, src_stoi, tgt_stoi, max_len=50)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
)

batch = next(iter(train_loader))
print("Batch shapes (src, tgt_in, tgt_out):", [x.shape for x in batch])

# Transformer-Modell

EMBED_SIZE = 128
NUM_HEADS = 4
FFN_HID_DIM = 512
NUM_ENCODER_LAYERS = 2
NUM_DECODER_LAYERS = 2
DROPOUT = 0.1

SRC_VOCAB_SIZE = len(src_itos)
TGT_VOCAB_SIZE = len(tgt_itos)

SRC_PAD_IDX = src_stoi[PAD_TOKEN]
TGT_PAD_IDX = tgt_stoi[PAD_TOKEN]

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)  
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0) 
        self.register_buffer("pe", pe)

    def forward(self, x):
    
        x = x + self.pe[:, : x.size(1), :]
        return self.dropout(x)

class Seq2SeqTransformer(nn.Module):
    def __init__(
        self,
        num_encoder_layers,
        num_decoder_layers,
        emb_size,
        nhead,
        src_vocab_size,
        tgt_vocab_size,
        dim_feedforward=512,
        dropout=0.1,
    ):
        super().__init__()

        self.src_tok_emb = nn.Embedding(src_vocab_size, emb_size, padding_idx=SRC_PAD_IDX)
        self.tgt_tok_emb = nn.Embedding(tgt_vocab_size, emb_size, padding_idx=TGT_PAD_IDX)
        self.positional_encoding = PositionalEncoding(emb_size, dropout=dropout)

        self.transformer = nn.Transformer(
            d_model=emb_size,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,  
        )

        self.generator = nn.Linear(emb_size, tgt_vocab_size)

    def forward(
        self,
        src,
        tgt,
        src_mask,
        tgt_mask,
        src_padding_mask,
        tgt_padding_mask,
        memory_key_padding_mask,
    ):
        
        src_emb = self.positional_encoding(self.src_tok_emb(src))  
        tgt_emb = self.positional_encoding(self.tgt_tok_emb(tgt))  

        outs = self.transformer(
            src_emb,
            tgt_emb,
            src_mask=src_mask,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_padding_mask,
            tgt_key_padding_mask=tgt_padding_mask,
            memory_key_padding_mask=memory_key_padding_mask,
        )  

        logits = self.generator(outs)  
        return logits

def generate_square_subsequent_mask(sz: int) -> torch.Tensor:
    mask = torch.triu(torch.ones((sz, sz)) * float("-inf"), diagonal=1)
    return mask  

def create_padding_mask(seq: torch.Tensor, pad_idx: int) -> torch.Tensor:
    return seq == pad_idx  

model = Seq2SeqTransformer(
    num_encoder_layers=NUM_ENCODER_LAYERS,
    num_decoder_layers=NUM_DECODER_LAYERS,
    emb_size=EMBED_SIZE,
    nhead=NUM_HEADS,
    src_vocab_size=SRC_VOCAB_SIZE,
    tgt_vocab_size=TGT_VOCAB_SIZE,
    dim_feedforward=FFN_HID_DIM,
    dropout=DROPOUT,
).to(DEVICE)

criterion = nn.CrossEntropyLoss(ignore_index=TGT_PAD_IDX)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)

print(
    "Modellparameter:",
    sum(p.numel() for p in model.parameters()) / 1e6,
    "Millionen",
)

# Training und Evaluation

NUM_EPOCHS = 10
train_losses = []
val_losses = []

def train_one_epoch(model, optimizer, dataloader):
    model.train()
    total_loss = 0.0

    for src_batch, tgt_in_batch, tgt_out_batch in dataloader:
        src_batch = src_batch.to(DEVICE)
        tgt_in_batch = tgt_in_batch.to(DEVICE)
        tgt_out_batch = tgt_out_batch.to(DEVICE)

        tgt_seq_len = tgt_in_batch.size(1)

        src_mask = None
        tgt_mask = generate_square_subsequent_mask(tgt_seq_len).to(DEVICE)

        src_padding_mask = create_padding_mask(src_batch, SRC_PAD_IDX).to(DEVICE)
        tgt_padding_mask = create_padding_mask(tgt_in_batch, TGT_PAD_IDX).to(DEVICE)

        logits = model(
            src_batch,
            tgt_in_batch,
            src_mask,
            tgt_mask,
            src_padding_mask,
            tgt_padding_mask,
            src_padding_mask,
        )

        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            tgt_out_batch.reshape(-1),
        )

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

    return total_loss / max(1, len(dataloader))

@torch.no_grad()
def evaluate(model, dataloader):
    model.eval()
    total_loss = 0.0

    for src_batch, tgt_in_batch, tgt_out_batch in dataloader:
        src_batch = src_batch.to(DEVICE)
        tgt_in_batch = tgt_in_batch.to(DEVICE)
        tgt_out_batch = tgt_out_batch.to(DEVICE)

        tgt_seq_len = tgt_in_batch.size(1)

        src_mask = None
        tgt_mask = generate_square_subsequent_mask(tgt_seq_len).to(DEVICE)

        src_padding_mask = create_padding_mask(src_batch, SRC_PAD_IDX).to(DEVICE)
        tgt_padding_mask = create_padding_mask(tgt_in_batch, TGT_PAD_IDX).to(DEVICE)

        logits = model(
            src_batch,
            tgt_in_batch,
            src_mask,
            tgt_mask,
            src_padding_mask,
            tgt_padding_mask,
            src_padding_mask,
        )

        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            tgt_out_batch.reshape(-1),
        )

        total_loss += loss.item()

    return total_loss / max(1, len(dataloader))

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\n=== EPOCH {epoch} ===")
    train_loss = train_one_epoch(model, optimizer, train_loader)
    val_loss = evaluate(model, val_loader)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

# Greedy Decoding und BLEU


@torch.no_grad()
def greedy_decode(model, src_sentence, max_len=50):
    model.eval()

    # EN-Satz zu IDs
    src_ids = encode_sentence(src_sentence, src_stoi, add_sos_eos=False)

    # Falls der Satz nach Tokenisierung leer ist → einfach leere Übersetzung zurückgeben
    if len(src_ids) == 0:
        return ""

    src_tensor = torch.tensor(src_ids, dtype=torch.long).unsqueeze(0).to(DEVICE)  
    src_padding_mask = create_padding_mask(src_tensor, SRC_PAD_IDX).to(DEVICE)

    # Encoder
    src_emb = model.positional_encoding(model.src_tok_emb(src_tensor))  
    memory = model.transformer.encoder(
        src_emb,
        src_key_padding_mask=src_padding_mask
    )  

    # Decoder-Start mit <sos>
    tgt_ids = [tgt_stoi[SOS_TOKEN]]

    for _ in range(max_len):
        tgt_tensor = torch.tensor(tgt_ids, dtype=torch.long).unsqueeze(0).to(DEVICE)  
        tgt_emb = model.positional_encoding(model.tgt_tok_emb(tgt_tensor))  

        tgt_mask = generate_square_subsequent_mask(tgt_tensor.size(1)).to(DEVICE)
        tgt_padding_mask = create_padding_mask(tgt_tensor, TGT_PAD_IDX).to(DEVICE)

        out = model.transformer.decoder(
            tgt=tgt_emb,
            memory=memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_padding_mask,
            memory_key_padding_mask=src_padding_mask
        )  

        logits = model.generator(out[:, -1, :])  # letztes Token
        next_token = logits.argmax(-1).item()
        tgt_ids.append(next_token)

        if next_token == tgt_stoi[EOS_TOKEN]:
            break

    # IDs zurück zu Tokens
    tokens = []
    for idx in tgt_ids[1:]:  # <sos> überspringen
        if idx == tgt_stoi[EOS_TOKEN]:
            break
        tokens.append(tgt_itos[idx])

    return " ".join(tokens)

@torch.no_grad()
def compute_bleu(model, val_en, val_de, max_samples=200):
    refs = []
    hyps = []

    n = min(len(val_en), max_samples)
    used = 0  # Anzahl tatsächlich genutzter Beispiele

    for i in range(n):
        src_sentence = val_en[i]
        tgt_sentence = val_de[i]

        # Leerzeilen überspringen
        if len(simple_tokenize(src_sentence)) == 0 or len(simple_tokenize(tgt_sentence)) == 0:
            continue

        hyp = greedy_decode(model, src_sentence)

        # Falls greedy_decode wegen leerem Satz "" zurückgibt, überspringen
        if len(simple_tokenize(hyp)) == 0:
            continue

        hyps.append(simple_tokenize(hyp))
        refs.append([simple_tokenize(tgt_sentence)])
        used += 1

        if used % 50 == 0:
            print(f"{used}/{n} gültige Beispiele übersetzt...")

    if used == 0:
        print("Keine gültigen Beispiele für BLEU (alle Sätze leer?)")
        return 0.0

    bleu = corpus_bleu(refs, hyps)
    return bleu

bleu_score = compute_bleu(model, val_en, val_de, max_samples=200)
print("BLEU-Score (Val-Subset):", bleu_score)

@torch.no_grad()
def show_translation_examples(model, val_en, val_de, num_examples=5):
    print("\n===== Beispielübersetzungen =====\n")
    for i in range(num_examples):
        src = val_en[i]
        tgt = val_de[i]
        pred = greedy_decode(model, src)

        print(f"Beispiel {i+1}:")
        print(f"EN:  {src}")
        print(f"DE (target): {tgt}")
        print(f"DE (pred):   {pred}")
        print("-" * 60)
show_translation_examples(model, val_en, val_de, num_examples=5)

# Loss-Kurve


plt.figure(figsize=(8, 5))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Train vs Validation Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()



### Additional Experiments *(5 additional points - <span style="color: red;">Optional</span>)*